# `test_eda.py` — Unit Tests for `ExploratoryAnalyzer`

## Purpose

Validates every public method of `ExploratoryAnalyzer`, ensuring turnover rate computations
are accurate, visualisation methods create output files, and the class never mutates its
input DataFrame.

---

## Module Under Test

`src.eda.exploratory_analyzer.ExploratoryAnalyzer`

---

## Test Classes at a Glance

| Class | Methods Tested | What It Verifies |
|-------|----------------|-----------------|
| `TestComputeTurnoverRate` | `compute_turnover_rate_by_feature()` | Return shape, column names, rate bounds, row sum, invalid feature error |
| `TestPlotDistribution` | `plot_distribution()` | File creation, invalid column error |
| `TestPlotCorrelationHeatmap` | `plot_correlation_heatmap()` | File creation |
| `TestPlotProjectCountBar` | `plot_project_count_bar()` | File creation |
| `TestInputImmutability` | all methods | Original DataFrame unchanged after analysis |

---

## Fixtures Used

| Fixture | Source |
|---------|--------|
| `sample_df` | `conftest.py` |
| `tmp_path` | pytest built-in |

---

## How to Run

```bash
pytest tests/test_eda.py -v
```


---

## `TestComputeTurnoverRate`

**Purpose:** Tests `compute_turnover_rate_by_feature()` — verifies the method groups employees
by a categorical feature, counts leavers per group, and calculates turnover rates that fall
within the valid [0, 1] range. Also checks that all 200 employees are accounted for.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_returns_dataframe` | Result is a `pd.DataFrame` |
| `test_has_required_columns` | Columns `count`, `left_count`, `turnover_rate` are present |
| `test_turnover_rates_between_0_and_1` | All `turnover_rate` values are in [0, 1] |
| `test_counts_sum_to_total_rows` | `count` column sums to `len(sample_df)` |
| `test_invalid_feature_raises` | Raises `ValueError` with "not in DataFrame" for unknown column |


In [ ]:
import pandas as pd
import pytest

from src.eda.exploratory_analyzer import ExploratoryAnalyzer
from src.utils.exceptions import VisualizationError


class TestComputeTurnoverRate:
    def test_returns_dataframe(self, sample_df):
        analyzer = ExploratoryAnalyzer(sample_df)
        result = analyzer.compute_turnover_rate_by_feature("salary")
        assert isinstance(result, pd.DataFrame)

    def test_has_required_columns(self, sample_df):
        analyzer = ExploratoryAnalyzer(sample_df)
        result = analyzer.compute_turnover_rate_by_feature("salary")
        assert "count" in result.columns
        assert "left_count" in result.columns
        assert "turnover_rate" in result.columns

    def test_turnover_rates_between_0_and_1(self, sample_df):
        analyzer = ExploratoryAnalyzer(sample_df)
        result = analyzer.compute_turnover_rate_by_feature("salary")
        assert result["turnover_rate"].between(0, 1).all()

    def test_counts_sum_to_total_rows(self, sample_df):
        analyzer = ExploratoryAnalyzer(sample_df)
        result = analyzer.compute_turnover_rate_by_feature("salary")
        assert result["count"].sum() == len(sample_df)

    def test_invalid_feature_raises(self, sample_df):
        analyzer = ExploratoryAnalyzer(sample_df)
        with pytest.raises(ValueError, match="not in DataFrame"):
            analyzer.compute_turnover_rate_by_feature("nonexistent_column")


---

## `TestPlotDistribution`

**Purpose:** Tests `plot_distribution()` — confirms the method writes a PNG file to the
specified path and raises a `ValueError` when an unknown column name is passed.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_invalid_column_raises_value_error` | Raises `ValueError` with "not found" for unknown column |
| `test_plot_creates_file` | PNG file exists at `save_path` after successful call |


In [ ]:
class TestPlotDistribution:
    def test_invalid_column_raises_value_error(self, sample_df, tmp_path):
        analyzer = ExploratoryAnalyzer(sample_df, output_dir=tmp_path)
        with pytest.raises(ValueError, match="not found"):
            analyzer.plot_distribution(
                column="nonexistent",
                title="Test",
            )

    def test_plot_creates_file(self, sample_df, tmp_path):
        analyzer = ExploratoryAnalyzer(sample_df, output_dir=tmp_path)
        analyzer.plot_distribution(
            column="satisfaction_level",
            title="Satisfaction Distribution",
            save_path=tmp_path / "test_dist.png",
        )
        assert (tmp_path / "test_dist.png").exists()


---

## `TestPlotCorrelationHeatmap` · `TestPlotProjectCountBar`

**Purpose:** Tests the remaining two visualisation methods — verifies that each writes its
output PNG to the specified `save_path`.

### Test Methods

| Method | Class | Verifies |
|--------|-------|----------|
| `test_heatmap_creates_file` | `TestPlotCorrelationHeatmap` | `test_heatmap.png` created |
| `test_bar_chart_creates_file` | `TestPlotProjectCountBar` | `test_bar.png` created |


In [ ]:
class TestPlotCorrelationHeatmap:
    def test_heatmap_creates_file(self, sample_df, tmp_path):
        analyzer = ExploratoryAnalyzer(sample_df, output_dir=tmp_path)
        analyzer.plot_correlation_heatmap(
            save_path=tmp_path / "test_heatmap.png"
        )
        assert (tmp_path / "test_heatmap.png").exists()


class TestPlotProjectCountBar:
    def test_bar_chart_creates_file(self, sample_df, tmp_path):
        analyzer = ExploratoryAnalyzer(sample_df, output_dir=tmp_path)
        analyzer.plot_project_count_bar(
            save_path=tmp_path / "test_bar.png"
        )
        assert (tmp_path / "test_bar.png").exists()


---

## `TestInputImmutability`

**Purpose:** Verifies that `ExploratoryAnalyzer` never modifies the DataFrame it receives.
This is a critical correctness property — downstream pipeline stages must receive the same
data that was passed in.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_original_df_not_mutated` | `sample_df` is identical before and after `compute_turnover_rate_by_feature()` |


In [ ]:
class TestInputImmutability:
    def test_original_df_not_mutated(self, sample_df):
        original_copy = sample_df.copy()
        analyzer = ExploratoryAnalyzer(sample_df)
        analyzer.compute_turnover_rate_by_feature("salary")
        pd.testing.assert_frame_equal(sample_df, original_copy)
